In [ ]:
# --------------------------------------------------------------
# Regras:
#  (1) IGUAIS (SEM PORT): igualdade literal de "Descricao Coletor"
#      apenas quando NENHUM lado contém "port" (case-insensitive).
#  (2) PORTS: quando ALGUM lado contém "port", o OUTRO também deve
#      conter "port" (case-insensitive), sem exigir igualdade literal.
# Em ambos os casos, aplica janela de horário <= 5 minutos.
# --------------------------------------------------------------

import re
import pandas as pd

# =================== CONFIG ===================
PARQUET_PATH = r"..\..\project\database\BaseGeral.parquet"
ID_A, ID_B   = "chapa1", "chapa2"
EXCLUDE_COLETOR = {"999"}
MATCH_SAME_DATE_EQ = True   # Regra (1): exigir mesma Data para iguais (recomendado)
PORT_TOKEN = "port"         # termo usado para identificar "portaria"

# ================ CARGA/FILTRO =================
df_ronda = pd.read_parquet(PARQUET_PATH)

base = df_ronda[
    df_ronda["Cadastro"].astype(str).isin([ID_A, ID_B])
    & ~df_ronda["Coletor"].astype(str).isin(EXCLUDE_COLETOR)
].copy()

A = base[base["Cadastro"].astype(str) == ID_A].copy().reset_index(drop=True)
B = base[base["Cadastro"].astype(str) == ID_B].copy().reset_index(drop=True)

# IDs únicos de linha (para segurança em anti-join, se precisarmos)
A["__rowA"] = A.index.astype(int)
B["__rowB"] = B.index.astype(int)

# Normalizações
for d in (A, B):
    d["Data"] = pd.to_datetime(d["Data"], errors="coerce").dt.date
    d["__desc_str"]   = d["Descricao Coletor"].astype(str)
    d["__desc_lower"] = d["__desc_str"].str.lower()
    d["__has_port"]   = d["__desc_lower"].str.contains(PORT_TOKEN, na=False)

def pick_nome(df):
    if "Nome" not in df.columns or df.empty:
        return "Pessoa"
    s = df["Nome"].dropna().astype(str)
    return s.iloc[-1] if not s.empty else "Pessoa"

def to_suffix(s):
    s = re.sub(r"\s+", "_", str(s).strip())
    s = re.sub(r"[^0-9A-Za-z_]+", "", s)
    return s or "Pessoa"

nome_a = to_suffix(pick_nome(A))
nome_b = to_suffix(pick_nome(B))

# ================ Helper janela de horário =================
def dif_minutos(df, col_a, col_b):
    hA = pd.to_datetime(df[col_a], format="%H:%M:%S", errors="coerce")
    hB = pd.to_datetime(df[col_b], format="%H:%M:%S", errors="coerce")
    na = hA.isna() | hB.isna()
    if na.any():
        hA.loc[na] = pd.to_datetime(df.loc[na, col_a], format="%H:%M", errors="coerce")
        hB.loc[na] = pd.to_datetime(df.loc[na, col_b], format="%H:%M", errors="coerce")
    return (hA - hB).abs().dt.total_seconds() / 60.0

col_hora_a = f"Hora_{nome_a}"
col_hora_b = f"Hora_{nome_b}"

# ================ (1) IGUAIS (SEM PORT) =====================
A_non = A[~A["__has_port"]].copy()
B_non = B[~B["__has_port"]].copy()

on_eq = ["Descricao Coletor"] + (["Data"] if MATCH_SAME_DATE_EQ else [])
pares_eq = A_non.merge(
    B_non,
    on=on_eq,
    how="inner",
    suffixes=(f"_{nome_a}", f"_{nome_b}")
)
pares_eq["dif_min"] = dif_minutos(pares_eq, col_hora_a, col_hora_b)
pares_eq = pares_eq[pares_eq["dif_min"] <= 5].copy()
pares_eq["RegraPareamento"] = "igual_descricao_sem_port"

# Guarda pares (A,B) que já entraram como iguais (não-port)
eq_pairs = set(pares_eq[["__rowA", "__rowB"]].apply(tuple, axis=1))

# ================ (2) PORTS ===============================
A_port = A[A["__has_port"]].copy()
B_port = B[B["__has_port"]].copy()

# Pareamento por Data (descrição pode diferir)
pares_port = A_port.merge(
    B_port,
    on=["Data"],
    how="inner",
    suffixes=(f"_{nome_a}", f"_{nome_b}")
)

pares_port["dif_min"] = dif_minutos(pares_port, col_hora_a, col_hora_b)
pares_port = pares_port[pares_port["dif_min"] <= 5].copy()

# Como os IGUAIS já excluem 'port', a interseção tende a ser nula.
# Ainda assim, garantimos que NÃO duplicaremos algum par por engano:
pairs_port_series = pares_port[["__rowA", "__rowB"]].apply(tuple, axis=1)
ports_final = pares_port[~pairs_port_series.isin(eq_pairs)].copy()
ports_final["RegraPareamento"] = "ambos_contem_port"

# ================ (3) UNIÃO FINAL =========================
resultado = pd.concat([pares_eq, ports_final], ignore_index=True, sort=False)

# Limpa auxiliares
drop_aux = [c for c in resultado.columns if c.startswith("__")]
resultado = resultado.drop(columns=drop_aux, errors="ignore")

# ================ SAÍDA ===================================
OUTPUT_PATH = r'.\comparacao_chapa1_chapa2.xlsx'
try:
    resultado.to_excel(OUTPUT_PATH, index=False)
    print(
        f"Arquivo salvo: {OUTPUT_PATH} | Linhas: {len(resultado)} "
        f"(iguais_sem_port={len(pares_eq)}, ports={len(ports_final)})"
    )
except Exception as e:
    print("Falha ao salvar Excel (ajuste OUTPUT_PATH).")
    print("Erro:", e)